In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_community.query_constructors.chroma import ChromaTranslator

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

In [3]:
# Movies dataset — rich, structured metadata makes self-query filtering meaningful
docs = [
    Document(
        page_content="A masked vigilante fights crime in a corrupt city with the help of a billionaire's technology. An iconic supervillain pushes him to his limits in a battle for Gotham's soul.",
        metadata={"title": "The Dark Knight", "genre": "action", "year": 2008, "rating": 9.0, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A thief who steals secrets through dream-sharing technology is offered a chance to have his past erased if he can plant an idea in someone's mind. A visually stunning exploration of the subconscious.",
        metadata={"title": "Inception", "genre": "sci-fi", "year": 2010, "rating": 8.8, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A team of explorers travels through a wormhole in space to find a new habitable planet for humanity. Stunning visuals of black holes and time dilation challenge our understanding of physics.",
        metadata={"title": "Interstellar", "genre": "sci-fi", "year": 2014, "rating": 8.6, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A programmer discovers that reality is a simulation and joins a rebellion against the machines controlling humanity. A groundbreaking blend of philosophy, martial arts, and bullet-time action.",
        metadata={"title": "The Matrix", "genre": "sci-fi", "year": 1999, "rating": 8.7, "director": "Lana Wachowski"},
    ),
    Document(
        page_content="Two criminals and a mob boss's wife are caught in a web of violence and dark humor over a single eventful day in Los Angeles. Interweaving storylines told out of chronological order.",
        metadata={"title": "Pulp Fiction", "genre": "drama", "year": 1994, "rating": 8.9, "director": "Quentin Tarantino"},
    ),
    Document(
        page_content="A maverick surgeon navigates the chaotic social landscape of a mobile army unit during the Korean War. Sharp satirical comedy disguised as a war film, later adapted into a beloved TV series.",
        metadata={"title": "MASH", "genre": "comedy", "year": 1970, "rating": 7.4, "director": "Robert Altman"},
    ),
    Document(
        page_content="Humanity sends a last-ditch mission to reignite the dying sun with a massive stellar bomb. An intense psychological thriller set in the terrifying emptiness of deep space.",
        metadata={"title": "Sunshine", "genre": "sci-fi", "year": 2007, "rating": 7.3, "director": "Danny Boyle"},
    ),
    Document(
        page_content="A soldier wakes up in another man's body aboard a commuter train just minutes before it explodes, reliving the event repeatedly to identify the bomber. A clever sci-fi thriller about time loops and identity.",
        metadata={"title": "Source Code", "genre": "sci-fi", "year": 2011, "rating": 7.5, "director": "Duncan Jones"},
    ),
]

print(f"Created {len(docs)} movie documents")

Created 8 movie documents


In [6]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="movie_collection"
)

In [9]:
# Defining Schema using AttributeInfo
metadata_field_info = [
    AttributeInfo(name="title", description="the title of movie", type="string"),
    AttributeInfo(name="genre", description="the genre of movie(action, sci-fi, drama, comedy)", type="string"),
    AttributeInfo(name="year", description="year the movie was released", type="integer"),
    AttributeInfo(name="rating", description="IMDb rating of movies (1-10)", type="float"),
    AttributeInfo(name="director", description="The director of the movie", type="string")
]

In [8]:
doc_content_description = "Brief plot description of movies"

In [12]:
retriever = SelfQueryRetriever.from_llm(
    vectorstore=vectorstore,
    llm=llm,
    document_contents=doc_content_description,
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator()
)

In [11]:
# create the vanilla vectorstore retriever
query = "What are some sci-fi movies released after 2010"
vs_retriever = vectorstore.as_retriever(search_type="similarity",
                                        search_kwargs={"k": 3})

results = vs_retriever.invoke(query)

for doc in results:
    print(f"[{doc.metadata['year']}] {doc.metadata['title']} ({doc.metadata['genre']}) - dir. {doc.metadata['director']}")
    print(f"  {doc.page_content[:100]}...")
    print()

[2007] Sunshine (sci-fi) - dir. Danny Boyle
  Humanity sends a last-ditch mission to reignite the dying sun with a massive stellar bomb. An intens...

[2014] Interstellar (sci-fi) - dir. Christopher Nolan
  A team of explorers travels through a wormhole in space to find a new habitable planet for humanity....

[1999] The Matrix (sci-fi) - dir. Lana Wachowski
  A programmer discovers that reality is a simulation and joins a rebellion against the machines contr...



In [ ]:
#SelfQuery Retriever:
query = "What are some sci-fi movies released after 2010"
results = retriever.invoke(query)

for doc in results:
    print(f"[{doc.metadata['year']}] {doc.metadata['title']} ({doc.metadata['genre']}) - dir. {doc.metadata['director']}")
    print(f"  {doc.page_content[:100]}...")
    print()

[2011] Source Code (sci-fi) - dir. Duncan Jones
  A soldier wakes up in another man's body aboard a commuter train just minutes before it explodes, re...

[2014] Interstellar (sci-fi) - dir. Christopher Nolan
  A team of explorers travels through a wormhole in space to find a new habitable planet for humanity....



In [17]:
# LLM extracts: semantic query="movies", filter={director: "Christopher Nolan"}

results = retriever.invoke("What movies did Christopher Nolan direct with above 8.7 ratings?")

for doc in results:
    print(f"[{doc.metadata['year']}] {doc.metadata['title']} - Rating: {doc.metadata['rating']}")
    print(f"  {doc.page_content[:100]}...")
    print()

[2010] Inception - Rating: 8.8
  A thief who steals secrets through dream-sharing technology is offered a chance to have his past era...

[2008] The Dark Knight - Rating: 9.0
  A masked vigilante fights crime in a corrupt city with the help of a billionaire's technology. An ic...

